In [1]:
#MHSA

import torch
import os
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader,random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from torch.cuda import manual_seed
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(path,transform = transform)

train_size = int(0.8*len(full_dataset))
val_size = int(0.1*len(full_dataset))
test_size = len(full_dataset) - train_size - val_size


train_dataset,val_dataset,test_dataset = random_split(
    full_dataset,
    [train_size,val_size,test_size],
    generator= torch.Generator().manual_seed(42)
)


Mounted at /content/drive


In [4]:
# Data Loader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False
)

In [5]:
#model

import torch
import torch.nn as nn

class MultiHeadSelfAttention(nn.Module):

  def __init__(self,emb_dim,num_heads):

    super().__init__()

    self.emb_dim = emb_dim
    self.num_heads = num_heads
    self.head_dim = emb_dim//num_heads

    assert(
        self.head_dim * num_heads== emb_dim
        ),"Embedding dimension must be divisible by heads"

    #Linear layer for query key value

    self.qkv = nn.Linear(emb_dim,emb_dim*3)

    #output projection

    self.fc_out = nn.Linear(emb_dim,emb_dim)

  def forward(self,query,key=None,value=None,need_weights=False):

    x=query

    batch_size,num_patch,em_dim = x.shape

    qkv = self.qkv(x)

    qkv = qkv.reshape(
        batch_size,
        num_patch,
        3,
        self.num_heads,
        self.head_dim
    )

    qkv = qkv.permute(2,0,3,1,4)

    queries = qkv[0]
    key = qkv[1]
    value = qkv[2]

    #Attention Score

    attention = torch.matmul(
        queries,
        key.transpose(-1,-2)
    )

    attention = attention / (self.head_dim)**0.5

    attention = torch.softmax(attention,dim=-1)

    #Weighted sum of values

    out = torch.matmul(attention,value)

    out = out.transpose(1,2).contiguous()

    out = out.reshape(
        batch_size,
        num_patch,
        self.emb_dim
    )

    out = self.fc_out(out)

    return out,attention


In [6]:
from torchvision.models import vit_b_16

model = vit_b_16(weights="IMAGENET1K_V1")

for block in model.encoder.layers:
  block.self_attention = MultiHeadSelfAttention(
      emb_dim=768,
      num_heads=12
  )

model.heads.head = nn.Linear(model.heads.head.in_features,2)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 153MB/s]


In [7]:
#Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [8]:
#training

epochs = 10
corrected = 0
totaL = 0
best_loss = float("inf")

for epoch in range(epochs):

  running_loss = 0

  model.train()

  for images, lables in train_loader:

    images = images.to(device)
    lables = lables.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs,lables)

    loss.backward()

    optimizer.step()

    running_loss +=loss.item()

  training_loss = running_loss / len(train_loader)

  model.eval()

  val_loss = 0

  with torch.no_grad():

    for images,labels in val_loader:

      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)

      loss = criterion(outputs,labels)

      val_loss += loss.item()
      _, predicated = torch.max(outputs,1)
      corrected+= (predicated == labels).sum().item()
      totaL+=labels.size(0)

    valdation_loss = val_loss / len(val_loader)
    val_accy = corrected / totaL *100

    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {training_loss:.4f}, Validation Loss: {valdation_loss:.4f},Validation acc: {val_accy:.2f}")

    if valdation_loss < best_loss:
      best_loss = valdation_loss
      torch.save(model.state_dict(),"best_model.pth")


Epoch 1/10, Training Loss: 0.6266, Validation Loss: 0.5778,Validation acc: 69.35
Epoch 2/10, Training Loss: 0.5488, Validation Loss: 0.3888,Validation acc: 78.14
Epoch 3/10, Training Loss: 0.5297, Validation Loss: 0.6776,Validation acc: 75.21
Epoch 4/10, Training Loss: 0.6058, Validation Loss: 0.5129,Validation acc: 75.00
Epoch 5/10, Training Loss: 0.5491, Validation Loss: 0.4002,Validation acc: 76.28
Epoch 6/10, Training Loss: 0.5512, Validation Loss: 0.4665,Validation acc: 76.05
Epoch 7/10, Training Loss: 0.5536, Validation Loss: 0.4288,Validation acc: 76.60
Epoch 8/10, Training Loss: 0.5131, Validation Loss: 0.3576,Validation acc: 78.08
Epoch 9/10, Training Loss: 0.4786, Validation Loss: 0.4153,Validation acc: 79.01
Epoch 10/10, Training Loss: 0.4583, Validation Loss: 0.5963,Validation acc: 77.44


In [9]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [10]:
#Accurcy

total=0
correct=0

with torch.no_grad():
  for images,labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    _, predicated = torch.max(output,1)

    total+=output.size(0)

    correct+=(predicated==labels).sum().item()

test_acc = 100*correct/total
print("Test accuracy: ",test_acc)



Test accuracy:  86.5
